In [ ]:
from seqfold import dg, fold, dot_bracket
import random

from Bio.Blast import NCBIXML
from Bio.SeqUtils import MeltingTemp as mt


def random_seq_list(length=20, num=50):
    nucleotides = ['A', 'T', 'C', 'G']
    sequence = [''.join(random.choice(nucleotides) for _ in range(length)) for j in range(num)]
    return sequence


def hum_dis(seq1, seq2):
    if len(seq1) != len(seq2):
        print("seq_not_match")
    else:
        cont = 0
        for char in range(len(seq1)):
            if seq1[char] != seq2[char]:
                cont += 1
        return cont


def calculate_dna_dna_tm(seq):
    """
    Calculate DNA/DNA hybridization melting temperature (Tm) using nearest-neighbor method.
    
    This function uses the BioPython MeltingTemp module with DNA_NN4 parameter table,
    which is suitable for DNA/DNA duplex melting temperature calculation.
    
    Args:
        seq: DNA sequence string (uppercase or lowercase)
        
    Returns:
        Melting temperature in degrees Celsius (float)
    """
    # Convert to uppercase to ensure consistency
    seq = seq.upper()
    # Use DNA_NN4 nearest-neighbor table for DNA/DNA hybridization
    # This uses default salt and DNA concentration parameters
    tm = mt.Tm_NN(seq, nn_table=mt.DNA_NN4)
    return tm


def dna_sec_struct(seq, temp=45):
    # Predict the minimum free energy
    mfe = dg(seq, temp=temp)
    # `fold` returns a list of `seqfold.Struct` from the minimum free energy structure
    structs = fold(seq, temp=temp)
    return mfe, structs


def thre_by_blast(file="./JXAKR9US016-Alignment.xml", thre=18):
    pos = []
    with open(file, "r") as blast_output:
        blast_records = NCBIXML.parse(blast_output)
        for blast_record in blast_records:
            save = True
            for alignment in blast_record.alignments:
                # print("Alignment title:", alignment.title)
                # print("Length of the alignment:", alignment.length)

                # # Iterate over the high-scoring pairs (HSPs) in the alignment
                for hsp in alignment.hsps:
                    # print("HSP score:", hsp.score)
                    if hsp.score >= thre:
                        pos.append(False)
                        save = False
                        break
                    # print("HSP bits:", hsp.bits)
                    # print("HSP query sequence:", hsp.query)
                    # print("HSP match sequence:", hsp.match)
                    # print("HSP subject sequence:", hsp.sbjct)
            if save:
                pos.append(True)
    return pos

In [ ]:
seq_list = random_seq_list(num=1000)

seq_list_export = []
for seq in seq_list:
    seq = seq.upper()

    # GGGGG
    if "GGGGG" in seq:
        print(f"{seq}: \tthre_by_G")
        continue

    # dif
    dif = True
    for tmp_seq in seq_list_export:
        if hum_dis(tmp_seq, seq) < 10:
            dif = False
            print(f"{seq}: \tthre_by_dif")
    if not dif:
        continue
    # # Calculate DNA/DNA melting temperature (Tm) and only keep sequences with Tm between 40-50°C
    # # Uses BioPython MeltingTemp module with DNA nearest-neighbor method for DNA/DNA hybridization
    # tm = calculate_dna_dna_tm(seq)
    # if not (40 <= tm <= 50):
    #     print(f"{seq}: \tthre_by_tm({tm:.2f}°C)")
    #     continue

    # secondary structure
    mfe, structs = dna_sec_struct(seq, temp=45)
    if mfe < 0:
        print(f"{seq}: \tthre_by_stru\t", dot_bracket(seq, structs))
        continue

    seq_list_export.append(seq)

print(f'{len(seq_list_export)} seqs remained')

In [ ]:
with open("./random_seq_filtered.txt", "w") as f:
    for _ in range(len(seq_list_export)):
        f.write(f'>seq{_}\n' + seq_list_export[_] + "\n")

In [ ]:
pos = thre_by_blast(file='./KDJF6467016-Alignment.xml', thre=18)

In [ ]:
with open("./random_seq_filtered.txt", "w") as f:
    cont = 0
    for _ in range(len(seq_list_export)):
        if pos[_]:
            f.write(f">seq{cont}\n" + seq_list_export[_] + "\n")
            cont += 1